# Analyzing Historical Stock & Revenue Data — TSLA vs. GME (Live Fetch)

**Objective:** Explore how **revenue** relates to **stock prices** for Tesla (TSLA) and GameStop (GME) using live data (no CSVs in repo).

**Pipeline**
1) Acquire (prices via API; revenue via `read_html`)  
2) Clean → standardize (`Date`, `Close`, `Revenue`, `Ticker`)  
3) Align revenue (quarterly) to price cadence  
4) Visualize & interpret (price trend, revenue trend, side-by-side)

**Reproducibility**
- All data fetched at runtime.
- Optional local cache (`USE_CACHE=True`) to freeze a run without committing files.

**Deliverables**
- Notebook with clear sections
- Exported figures in `../figures/`


In [2]:
pip install yfinance

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.2/949.2 kB 6.3 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.7 MB/s  0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15636 sha256=3c7612b77e744d58513d99bd8035ad4b9da206553054f944da0b38ed8cce6ae5
  Stored in directory: /Users/leonagarabedian/Library/Caches/pip/wheels/e9/25/85/25d2e1cfc0ece64b930b16972f7e4cc3599c43b531f1eba06d
  Created wheel for peewee: filename=peewee-3.18.2-cp310-cp310-macosx_10_9_universal2.whl size=415717 sha256=9b92c8ef441ca3bac3d2cc08d01e4d9f1e88c60c720afc480ddd2a5a6b376ee1
  Stored in directory: /Users/leonagarabedian/Library/Caches/pip/wheels/29/22/6c/745744e946d21fdb

In [1]:
from __future__ import annotations
import os, time, io
from datetime import datetime
import pandas as pd
import numpy as np
import requests
import yfinance as yf

# ---------- Configuration ----------
TICKERS = ["TSLA", "GME"]

REVENUE_URLS = {
    "TSLA": "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue",
    "GME":  "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue",
}

USE_CACHE = False            # keep False to avoid writing files by default
CACHE_DIR = "../data/cache"  # ignored by git via .gitignore
os.makedirs(CACHE_DIR, exist_ok=True)

def _cache_path(name: str) -> str:
    return os.path.join(CACHE_DIR, f"{name}.parquet")

def _maybe_cache(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if USE_CACHE:
        df.to_parquet(_cache_path(name), index=False)
    return df

def _maybe_read_cache(name: str) -> pd.DataFrame | None:
    path = _cache_path(name)
    return pd.read_parquet(path) if (USE_CACHE and os.path.exists(path)) else None


ModuleNotFoundError: No module named 'yfinance'

In [ ]:
# === 1) Acquire prices (yfinance) ===
def fetch_prices(ticker: str, period="max", interval="1d") -> pd.DataFrame:
    cached = _maybe_read_cache(f"{ticker}_price")
    if cached is not None:
        return cached

    df = yf.download(ticker, period=period, interval=interval, progress=False)
    if df.empty:
        raise ValueError(f"No price data for {ticker}.")
    df = df.reset_index().rename(columns={"Date": "Date", "Close": "Close"})
    df = df[["Date", "Close"]].assign(Ticker=ticker)
    return _maybe_cache(df, f"{ticker}_price")

prices = pd.concat([fetch_prices(t) for t in TICKERS], ignore_index=True)
prices.sort_values(["Ticker", "Date"]).head()


In [ ]:
# === 2) Acquire revenue (read_html) ===
import io
def fetch_revenue_quarterly(ticker: str) -> pd.DataFrame:
    cached = _maybe_read_cache(f"{ticker}_revenue")
    if cached is not None:
        return cached

    url = REVENUE_URLS[ticker]
    resp = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    tables = pd.read_html(io.StringIO(resp.text))
    cand = None
    for tbl in tables:
        cols = [c.strip().lower() for c in map(str, tbl.columns)]
        if any("quarter" in c for c in cols) and any("revenue" in c for c in cols):
            cand = tbl
            break
    if cand is None:
        raise ValueError(f"Revenue table not found for {ticker} at {url}")

    cand.columns = [c.strip() for c in cand.columns]
    qcol = [c for c in cand.columns if "Quarter" in c or "quarter" in c][0]
    rcol = [c for c in cand.columns if "Revenue" in c or "revenue" in c][0]

    df = cand[[qcol, rcol]].copy().rename(columns={qcol: "Period", rcol: "Revenue"})
    df["Revenue"] = (
        df["Revenue"].astype(str).str.replace(r"[\$,]", "", regex=True)
        .replace({"": np.nan, "None": np.nan, "nan": np.nan})
        .astype(float)
    )
    # Try direct parsing (e.g., 2019-03-31)
    df["Date"] = pd.to_datetime(df["Period"], errors="coerce")
    if df["Date"].isna().all():
        # Fallback for formats like "2019 Q1"
        df[["Year", "Q"]] = df["Period"].str.extract(r"(\d{4})\s*Q?(\d)")
        df["Date"] = pd.PeriodIndex(year=df["Year"].astype(int),
                                    quarter=df["Q"].astype(int),
                                    freq="Q").end_time
    df = df.dropna(subset=["Date"]).sort_values("Date")
    df = df[["Date", "Revenue"]].assign(Ticker=ticker)
    return _maybe_cache(df, f"{ticker}_revenue")

revenue = pd.concat([fetch_revenue_quarterly(t) for t in TICKERS], ignore_index=True)
revenue.sort_values(["Ticker", "Date"]).head()


In [ ]:
# === 3) Align quarterly revenue to daily prices ===
prices_sorted = prices.sort_values(["Ticker", "Date"]).copy()
rev_sorted    = revenue.sort_values(["Ticker", "Date"]).copy()

aligned = pd.merge_asof(
    prices_sorted,
    rev_sorted,
    by="Ticker",
    on="Date",
    direction="backward"   # take most recent revenue reported before/at the price date
)
aligned.head()


In [ ]:
# === 4) Visualize & export figures ===
import matplotlib.pyplot as plt
import os
os.makedirs("../figures", exist_ok=True)

def plot_price(df, ticker, outfile):
    d = df[df["Ticker"]==ticker]
    ax = d.plot(x="Date", y="Close", figsize=(10,5), title=f"{ticker} Closing Price")
    ax.figure.tight_layout()
    ax.figure.savefig(outfile, dpi=180)
    plt.close(ax.figure)

plot_price(prices, "TSLA", "../figures/tsla_price.png")
plot_price(prices, "GME", "../figures/gme_price.png")

def plot_price_vs_rev(df, ticker, outfile):
    d = df[df["Ticker"]==ticker].dropna(subset=["Revenue"])
    fig, ax1 = plt.subplots(figsize=(11,5))
    ax1.plot(d["Date"], d["Close"], label="Close")
    ax1.set_ylabel("Close (USD)")
    ax1.set_title(f"{ticker}: Price vs. Quarterly Revenue (aligned)")
    ax2 = ax1.twinx()
    ax2.plot(d["Date"], d["Revenue"], label="Revenue", linestyle="--")
    ax2.set_ylabel("Revenue (USD)")
    fig.tight_layout()
    fig.savefig(outfile, dpi=180)
    plt.close(fig)

plot_price_vs_rev(aligned, "TSLA", "../figures/price_vs_revenue_tsla.png")
plot_price_vs_rev(aligned, "GME",  "../figures/price_vs_revenue_gme.png")
print("Saved figures to ../figures")


In [ ]:
# === 5) Summary (concise, recruiter-friendly) ===
summary = {
    "TSLA": "Revenue expansion (2019–2021+) broadly coincides with sustained price strength.",
    "GME":  "Event-driven price spikes decouple from revenue fundamentals."
}
for k,v in summary.items():
    print(f"{k}: {v}")
